1. Read file_gpkg (This is generated by data_processing.ipynb)
2. Count the number of rows (PV_normal, PV_pool, PV_heater)
    - Some rows corresponds to multiple columns. (e.g. PV_pool & uncertflag)
3. Select Sample rows and saved it as 'sample_annotations_for_qc'

In [2]:
import geopandas as gpd
final_annotation = "../1.db_pipeline/final_annotations.gpkg"
gdf = gpd.read_file(final_annotation)
gdf

print(gdf.dtypes)

id                                 int64
PV_normal                        float64
PV_heater                        float64
PV_pool                          float64
uncertflag                       float64
area                             float64
annotator                         object
centroid_latitude                float64
centroid_longitude               float64
image_name                        object
nw_corner_of_image_latitude       object
nw_corner_of_image_longitude      object
se_corner_of_image_latitude       object
se_corner_of_image_longitude      object
geometry                        geometry
dtype: object


In [9]:
print(f"The number of annotations is {len(gdf['id'])}")

columns_to_check = ['PV_normal', 'PV_heater', 'PV_pool', 'uncertflag']
counts = (gdf[columns_to_check] == 1.0).sum()
print(counts)


only_uncertflag = gdf[
    (gdf['uncertflag'] == 1.0) &
    (gdf[['PV_normal', 'PV_heater', 'PV_pool']] != 1.0).all(axis=1)
]
print(f"The number of annotations with only 'uncertflag' = 1.0 (no double counts): {len(only_uncertflag)}")


The number of annotations is 19735
PV_normal     10787
PV_heater      5216
PV_pool        2503
uncertflag     1920
dtype: int64
The number of annotations with only 'uncertflag' = 1.0 (no double counts): 1231


**Count rows corresponding to multiple columns**
 - PV_pool & uncertflag = 205
 - PV_heater & uncertflag = 484
 - PV_heater & PV_pool = 2 

In [10]:
import pandas as pd
from itertools import combinations

# Define the columns to check
columns_to_check = ['PV_normal', 'PV_heater', 'PV_pool', 'uncertflag']

# Dictionary to store results
combo_counts = {}

# Count for each combination of size >= 2
for r in range(2, len(columns_to_check) + 1):
    for combo in combinations(columns_to_check, r):
        condition = (gdf[list(combo)] == 1.0).all(axis=1)
        count = condition.sum()
        combo_counts[combo] = count

# Display results
for combo, count in combo_counts.items():
    print(f"{' & '.join(combo)} = {count}")


PV_normal & PV_heater = 0
PV_normal & PV_pool = 0
PV_normal & uncertflag = 0
PV_heater & PV_pool = 2
PV_heater & uncertflag = 484
PV_pool & uncertflag = 205
PV_normal & PV_heater & PV_pool = 0
PV_normal & PV_heater & uncertflag = 0
PV_normal & PV_pool & uncertflag = 0
PV_heater & PV_pool & uncertflag = 0
PV_normal & PV_heater & PV_pool & uncertflag = 0


Note (June.16) 
- Drops 'uncertflag', & double annotation (PV_heater & PV_pool)
- 17,813 annotations (PV_normal: 10787, PV_heater: 4730, PV_pool: 2296)
- Stratified sample of 1,000 across classes and divides it into four

In [8]:
# Imports final_annotations.gpkg to keep embedded CRS

final_annotation = "../1.db_pipeline/final_annotations.gpkg"
final_annotation = gpd.read_file(final_annotation)
cond_uncert_not_1 = final_annotation['uncertflag'] != 1.0
cond_not_both_heater_pool = (final_annotation['PV_heater'] != 1.0) | (final_annotation['PV_pool'] != 1.0)
filtered_gdf = final_annotation.loc[cond_uncert_not_1 & cond_not_both_heater_pool]

columns_to_check = ['PV_normal', 'PV_heater', 'PV_pool']
counts = (filtered_gdf[columns_to_check] == 1.0).sum()
print(counts)

filtered_gdf

PV_normal    10787
PV_heater     4730
PV_pool       2296
dtype: int64


,id,PV_normal,PV_heater,PV_pool,uncertflag,area,annotator,centroid_latitude,centroid_longitude,image_name,nw_corner_of_image_latitude,nw_corner_of_image_longitude,se_corner_of_image_latitude,se_corner_of_image_longitude,geometry
0,1,1.0,NaN,NaN,NaN,23.162657,biz,-3.769763e+06,-19991.996891,2023_RGB_8cm_W16C_21,-3769000.0,-20000.0,-3770000.0,-19000.0,"POLYGON ((-19993.55 -3769759.6, -19988.801 -37..."
1,2,NaN,NaN,1.0,NaN,22.464224,biz,-3.775459e+06,-7333.463276,2023_RGB_8cm_W07C_3,-3775000.0,-8000.0,-3776000.0,-7000.0,"POLYGON ((-7338.039 -3775460.804, -7330.294 -3..."
2,3,NaN,NaN,1.0,NaN,71.734771,biz,-3.775556e+06,-4412.644364,2023_RGB_8cm_W07D_1,-3775000.0,-5000.0,-3776000.0,-4000.0,"POLYGON ((-4418.027 -3775551.846, -4406.117 -3..."
3,4,NaN,NaN,1.0,NaN,16.578681,biz,-3.780474e+06,-9142.853580,2023_RGB_8cm_W08A_1,-3780000.0,-10000.0,-3781000.0,-9000.0,"POLYGON ((-9145.183 -3780476.685, -9142.692 -3..."
4,5,NaN,NaN,1.0,NaN,27.684287,biz,-3.780033e+06,-8917.850802,2023_RGB_8cm_W08A_2,-3780000.0,-9000.0,-3781000.0,-8000.0,"POLYGON ((-8922.826 -3780035.448, -8914.535 -3..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19730,19731,1.0,NaN,NaN,NaN,13.599248,mejia,-3.758774e+06,-49022.331892,2023_RGB_8cm_W45C_16,-3758000.0,-50000.0,-3759000.0,-49000.0,"POLYGON ((-49023.481 -3758771.941, -49020.398 ..."
19731,19732,1.0,NaN,NaN,NaN,4.845304,mejia,-3.758791e+06,-49072.211780,2023_RGB_8cm_W45C_16,-3758000.0,-50000.0,-3759000.0,-49000.0,"POLYGON ((-49074.803 -3758789.825, -49069.39 -..."
19732,19733,1.0,NaN,NaN,NaN,4.371103,mejia,-3.758792e+06,-49072.717572,2023_RGB_8cm_W45C_16,-3758000.0,-50000.0,-3759000.0,-49000.0,"POLYGON ((-49075.145 -3758791.263, -49069.938 ..."
19733,19734,1.0,NaN,NaN,NaN,4.169215,mejia,-3.758794e+06,-49072.796580,2023_RGB_8cm_W45C_16,-3758000.0,-50000.0,-3759000.0,-49000.0,"POLYGON ((-49075.214 -3758792.771, -49070.075 ..."


In [ ]:
import pandas as pd

total = counts.sum()
sample_counts = (counts / total * 1000).round().astype(int)

pv_normal_df = filtered_gdf[filtered_gdf['PV_normal'] == 1.0]
pv_heater_df = filtered_gdf[filtered_gdf['PV_heater'] == 1.0]
pv_pool_df = filtered_gdf[filtered_gdf['PV_pool'] == 1.0]

# Fix random_state for 
sampled_normal = pv_normal_df.sample(n=sample_counts['PV_normal'], random_state=42)
sampled_heater = pv_heater_df.sample(n=sample_counts['PV_heater'], random_state=42)
sampled_pool = pv_pool_df.sample(n=sample_counts['PV_pool'], random_state=42)

final_sampled_df = pd.concat([sampled_normal, sampled_heater, sampled_pool], ignore_index=True)
final_sampled_df.shape

final_sampled_df

,id,PV_normal,PV_heater,PV_pool,uncertflag,area,annotator,centroid_latitude,centroid_longitude,image_name,nw_corner_of_image_latitude,nw_corner_of_image_longitude,se_corner_of_image_latitude,se_corner_of_image_longitude,geometry
0,15090,1.0,NaN,NaN,NaN,9.470225,mukhtar,-3.757600e+06,-27944.750907,2023_RGB_8cm_W25C_13,-3757000.0,-28000.0,-3758000.0,-27000.0,"POLYGON ((-27947.237 -3757599.612, -27946.392 ..."
1,19406,1.0,NaN,NaN,NaN,2.798719,veen,-3.742493e+06,-28673.983328,2023_RGB_8cm_W24A_12,-3742000.0,-29000.0,-3743000.0,-28000.0,"POLYGON ((-28675.187 -3742492.981, -28673.661 ..."
2,6398,1.0,NaN,NaN,NaN,3.797902,fiona,-3.775313e+06,-58266.106395,2023_RGB_8cm_W57C_2,-3775000.0,-59000.0,-3776000.0,-58000.0,"POLYGON ((-58267.414 -3775312.354, -58265.403 ..."
3,8478,1.0,NaN,NaN,NaN,5.857445,ye,-3.781234e+06,-11074.026181,2023_RGB_8cm_W18B_9,-3781000.0,-12000.0,-3782000.0,-11000.0,"POLYGON ((-11077.105 -3781232.642, -11070.849 ..."
4,14287,1.0,NaN,NaN,NaN,37.251612,mukhtar,-3.755277e+06,-29918.944319,2023_RGB_8cm_W25C_1,-3755000.0,-30000.0,-3756000.0,-29000.0,"POLYGON ((-29928.266 -3755274.267, -29909.129 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,11059,NaN,NaN,1.0,NaN,17.469904,mittal,-3.731472e+06,-51479.897963,2023_RGB_8cm_W53B_9,-3731000.0,-52000.0,-3732000.0,-51000.0,"POLYGON ((-51480.763 -3731469.003, -51477.895 ..."
997,3865,NaN,NaN,1.0,NaN,18.951225,fiona,-3.770877e+06,-51052.166679,2023_RGB_8cm_W57B_4,-3770000.0,-52000.0,-3771000.0,-51000.0,"POLYGON ((-51055.337 -3770876.193, -51050.026 ..."
998,3808,NaN,NaN,1.0,NaN,23.087963,fiona,-3.770713e+06,-51883.898391,2023_RGB_8cm_W57B_4,-3770000.0,-52000.0,-3771000.0,-51000.0,"POLYGON ((-51888.437 -3770710.924, -51879.133 ..."
999,3771,NaN,NaN,1.0,NaN,9.030946,fiona,-3.770795e+06,-51223.571202,2023_RGB_8cm_W57B_4,-3770000.0,-52000.0,-3771000.0,-51000.0,"POLYGON ((-51225.948 -3770794.952, -51224.877 ..."


In [9]:
chunk_size = final_sampled_df.shape[0] // 4

df_part1 = final_sampled_df.iloc[:chunk_size].reset_index(drop=True)
df_part2 = final_sampled_df.iloc[chunk_size:chunk_size*2].reset_index(drop=True)
df_part3 = final_sampled_df.iloc[chunk_size*2:chunk_size*3].reset_index(drop=True)
df_part4 = final_sampled_df.iloc[chunk_size*3:].reset_index(drop=True)

print(df_part1.shape, df_part2.shape, df_part3.shape, df_part4.shape)

df_part1 = gpd.GeoDataFrame(df_part1, geometry='geometry')
df_part2 = gpd.GeoDataFrame(df_part2, geometry='geometry')
df_part3 = gpd.GeoDataFrame(df_part3, geometry='geometry')
df_part4 = gpd.GeoDataFrame(df_part4, geometry='geometry')

df_part1.to_file("sampled_part1_Shawn.gpkg", driver="GPKG")
df_part2.to_file("sampled_part2_Priyanka.gpkg", driver="GPKG")
df_part3.to_file("sampled_part3_Catherine.gpkg", driver="GPKG")
df_part4.to_file("sampled_part4_Rema.gpkg", driver="GPKG")


(250, 15) (250, 15) (250, 15) (251, 15)
